In [ ]:
!pip install -q langchain-chroma langchain_community langchain-huggingface sentence_transformers langgraph langchain-google-genai  rank_bm25

In [ ]:
from google.colab import userdata
from google.colab import userdata

In [ ]:
import os
import regex
import json
from IPython.display import Markdown
from sqlalchemy import create_engine
import pandas as pd

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model
from langchain_community.retrievers import BM25Retriever, En

In [ ]:
# Env vars
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = "/content/drive/MyDrive/cs431_data/outputs_processed/transcripts"

In [ ]:
video_names = set()

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".json"):
          name = file.replace("_ocr.json", "")
          name = name.replace("_asr.json", "")
          video_names.add(name)

In [ ]:
for i in video_names:
  print(i)

[CS431 - Chương 7] Part 1_ Giới thiệu mạng RNN - Xử lý dữ liệu dạng chuỗi
[CS431 - Chương 8] Part 2_ Một số biến thể của RNN_ Bidirectional RNN
[CS431 - Chương 8] Part 1_1_ Một số biến thể của RNN_ LSTM
[CS431 - Chương 2] Part 4a_ Mô hình hồi quy Softmax (SoftmaxRegression)
[CS431 - Chương 9] Part 1_1_ Giới thiệu bài toán Dịch máy
[CS431 - Chương 6] Part 2_ Hướng tiếp cận Deep learning cho các bài toán NLP
[CS431 - Chương 8] Part 1_2_ Một số biến thể của RNN_ LSTM
[CS431 - Các kĩ thuật học sâu và ứng dụng] Video 1.3.2 Ôn tập nền tảng đại số tuyến tính (Part 2)
[CS431 - Chương 10] Part 1_ Ôn tập kiến trúc mạng RNN và cơ chế Attention
[CS431 - Chương 10] Part 4_3_ Kiến trúc Transformer_ Bộ Encoder
[CS431 - Chương 2] Part 2a_1_ Mô hình hồi quy tuyến tính (Linear Regression) - Trường Đại học Công nghệ Thông tin - UIT
[CS431 - Chương 5] Part 4_ Ứng du

## Fusion

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
)

def fuse_and_refine_asr_ocr(asr_data, ocr_data, video_name, llm):

    raw_documents = []

    # --- BƯỚC 1: GỘP DỮ LIỆU THÔ ---
    for i in range(len(ocr_data)):
        current_ocr = ocr_data[i]
        start_time = current_ocr['ts']

        if i + 1 < len(ocr_data):
            end_time = ocr_data[i+1]['ts']
        else:
            end_time = asr_data[-1]['end']

        # Tìm ASR trong khoảng thời gian (bao gồm cả buffer trước/sau 1 câu)
        inside_indices = [
            j for j, item in enumerate(asr_data)
            if start_time <= item['start'] < end_time
        ]

        relevant_asr = ""
        if inside_indices:
            start_idx = max(0, inside_indices[0] - 1)
            end_idx = min(len(asr_data) - 1, inside_indices[-1] + 1)
            relevant_asr = " ".join([asr_data[idx]['text'] for idx in range(start_idx, end_idx + 1)])

        # Lưu tạm vào list Document thô
        fused_text_raw = f"NỘI DUNG SLIDE: {current_ocr['text']}\nNỘI DUNG GIẢNG VIÊN NÓI: {relevant_asr}"

        raw_documents.append(
            Document(
                page_content=fused_text_raw,
                metadata={
                    "timestamp": start_time,
                    "duration": end_time - start_time,
                    "video_name": video_name
                }
            )
        )

    # --- BƯỚC 2: TIẾN HÀNH REFINE BẰNG LLM ---
    refined_results = []
    print(f"Đang refine {len(raw_documents)} docs...")

    for i in range(len(raw_documents)):
        # Lấy ngữ cảnh trước và sau (nếu có)
        prev_content = raw_documents[i-1].page_content if i > 0 else "Không có dữ liệu (Bắt đầu video)"
        current_content = raw_documents[i].page_content
        next_content = raw_documents[i+1].page_content if i < len(raw_documents) - 1 else "Không có dữ liệu (Kết thúc video)"

        context_prompt = f"""
Dưới đây là nội dung bài giảng tại 3 thời điểm liên tiếp (bao gồm Slide và Lời giảng):

[TRƯỚC ĐÓ]
{prev_content}

[HIỆN TẠI - CẦN TINH CHỈNH TOÀN BỘ]
{current_content}

[TIẾP THEO]
{next_content}

### Nhiệm vụ:
Dựa vào ngữ cảnh trước và sau, hãy biên tập lại toàn bộ nội dung của thời điểm [HIỆN TẠI] bao gồm cả phần nội dung Slide và Lời giảng của giảng viên.

### Yêu cầu cụ thể:
1. **Đối với Nội dung Slide:**
   - Sửa các lỗi chính tả do nhận diện hình ảnh (OCR) sai.
   - Trình bày lại các ý trên slide dưới dạng danh sách (bullet points) để rõ ràng, rành mạch.
   - Đảm bảo các thuật ngữ trên Slide đồng nhất với Lời giảng.

2. **Đối với Lời giảng của Giảng viên:**
   - Khắc phục lỗi nhận diện giọng nói (ASR) và các từ nối thừa (à, ừm...).
   - Đảm bảo câu văn trôi chảy, có sự kết nối logic với phần [TRƯỚC ĐÓ] và dẫn dắt mượt mà sang phần [TIẾP THEO].
   - Giữ nguyên phong cách của giảng viên nhưng làm cho câu từ gãy gọn hơn.

3. **Định dạng đầu ra (MARKDOWN):**
   - Sử dụng tiêu đề `### Nội dung Slide` và `### Lời giảng của Giảng viên`.
   - **Bôi đậm (bold)** các thuật ngữ chuyên môn hoặc từ khóa quan trọng.
   - Sử dụng các ký hiệu Markdown (-, *, 1., 2.) để phân tách các ý.
   - **Chỉ trả về duy nhất nội dung đã tinh chỉnh**, không có lời dẫn giải thêm.
"""
        # Gọi LLM
        try:
            refined_content = llm.invoke(context_prompt).content # hoặc .page_content tùy loại LLM

            # Tạo Document mới với nội dung đã sạch
            refined_results.append(
                Document(
                    page_content=refined_content,
                    metadata=raw_documents[i].metadata
                )
            )
        except Exception as e:
            print(f"Lỗi tại index {i}: {e}")
            # Nếu lỗi thì giữ nguyên nội dung thô để tránh mất dữ liệu
            refined_results.append(raw_documents[i])

    return refined_results

In [ ]:
documents = []

db_file_path = '/content/drive/MyDrive/cs431_data/database/documents.db'
engine = create_engine(f'sqlite:///{db_file_path}')

if not os.path.exists(db_file_path): # Neu khong co documents db (doc chua duoc xu ly va luu tru)

  for name in video_names:
    ocr_trans = json.load(open(os.path.join(data_path, name + "_ocr.json"), "r"))
    asr_trans = json.load(open(os.path.join(data_path, name + "_asr.json"), "r"))

    ocr_trans.sort(key=lambda x: x['ts'])
    asr_trans.sort(key=lambda x: x['start'])

    final_chunks = fuse_and_refine_asr_ocr(asr_trans, ocr_trans, name, llm)

    documents.extend(final_chunks)

    # Prepare data for insertion
    data_to_save = []
    for doc in documents:
        data_to_save.append({
            'video_name': doc.metadata.get('video_name'),
            'timestamp': doc.metadata.get('timestamp'),
            'duration': doc.metadata.get('duration'),
            'content': doc.page_content
        })

    # Create a DataFrame
    df_docs = pd.DataFrame(data_to_save)

    # Connect to SQLite and save
    df_docs.to_sql('documents', con=engine, if_exists='replace', index=False)

    print("Saved documents to DB")
else: # Neu document da duoc xu ly va luu trong db
  docs = pd.read_sql('SELECT * FROM documents', con=engine)

  for i, row in docs.iterrows():
    documents.append(
        Document(
            page_content=row['content'],
            metadata={
                'video_name': row['video_name'],
                'timestamp': row['timestamp'],
                'duration': row['duration']
            }
        )
    )

  print(f"Loaded documents from DB: {len(documents)} rows")


Loaded documents from DB: 757 rows


In [ ]:
Markdown(documents[100].page_content)

### Nội dung Slide

*   **DeepLab-v3**
*   **Phép convolution**
*   **Phép Atrous/Dilated convolution**
*   Kiến trúc mạng **DeepLab-v3**

### Lời giảng của Giảng viên

Một kiến trúc khác cũng rất nổi tiếng, đó chính là **DeepLab-v3**. Ý tưởng của **DeepLab-v3** dựa trên phép tính toán gọi là **Atrous Convolution**, hay còn có tên gọi khác là **Dilated Convolution**.

Nếu như phép **Convolution** thông thường, ví dụ đây là output, đây là input, nó sẽ tổng hợp thông tin của một vùng có kích thước 3x3 để điền vào một điểm trên feature map. Việc này dẫn đến vấn đề là không tổng hợp được thông tin ở những vùng có kích thước lớn hơn. Muốn tổng hợp được thông tin ở những vùng lớn hơn, chúng ta sẽ phải thực hiện phép **convolution** liên tiếp nhiều lần. Điều này làm tăng chi phí tính toán và không giải quyết được vấn đề về scale, vấn đề về độ bất biến trong phân đoạn ngữ nghĩa với các đối tượng nhỏ hoặc rất lớn.

Trong khi đó, phép **Atrous Convolution** thay vì lấy các vùng 3x3 liên tiếp nhau, chúng ta có thể **skip** (bỏ qua) các ô ở giữa. Trong ví dụ này, **rate** (tốc độ/bước nhảy) thể hiện mức độ **skip**. Ở đây, **rate** bằng 2, nghĩa là bỏ qua 2 ô. Còn ở ví dụ kia, **rate** bằng 1, tức là chỉ nhảy cóc 1 đơn vị để lấy giá trị tổng hợp thông tin. Phép **Atrous Convolution** là sự kết hợp thông tin tại rất nhiều vùng với các kích thước khác nhau, ví dụ từ một vùng 1x1, từ một vùng 3x3. Thực ra cái 1x1 convolution này...

In [ ]:
documents[100].metadata

{'video_name': '[CS431 - Chương 5] Part 4_ Ứng dụng mạng CNN cho bài toán Phân đoạn ngữ nghĩa đối tượng',
 'timestamp': 400,
 'duration': 20.0}

## Vector DB, Embedding, Saving

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector = embeddings.embed_query("hello, world!")
len(vector)

3072

In [ ]:
vector_db_file_path = "/content/drive/MyDrive/cs431_data/database/chroma_langchain_db_google_embedding"

vector_store = Chroma(
    collection_name="default",
    embedding_function=embeddings,
    persist_directory=vector_db_file_path)


In [ ]:
# if not os.path.exists(vector_db_file_path):
# vector_store.add_documents(documents)

## Retrieval Testing

In [ ]:
semantic_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})
bm25_retriever = BM25Retriever.from_documents(documents)

def hybrid_search(query, vectordb, bm25_retriever, k=5):
    # Semantic Search (tao lai retriever tu dau)
    semantic_results = vectordb.similarity_search(query=query, k=k*2)

    # BM25 Search
    bm25_retriever.k = k*2
    bm25_results = bm25_retriever.invoke(query)

    # Gộp và xếp hạng bằng Reciprocal Rank Fusion
    rrf_scores = {}

    def add_results_to_rrf(results, weight=1.0):
        for rank, doc in enumerate(results):
            # Dùng nội dung text làm khóa để deduplicate
            doc_id = doc.page_content

            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = {"doc": doc, "score": 0.0}

            # Tính điểm RRF
            rrf_scores[doc_id]["score"] += weight * (1.0 / (rank + 1 + 60))

    # Đưa kết quả của cả 2 phương pháp vào tính điểm
    add_results_to_rrf(semantic_results)
    add_results_to_rrf(bm25_results)

    # Sắp xếp các document theo điểm RRF giảm dần
    reranked_docs = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)

    # Lấy Top K tài liệu xuất sắc nhất
    final_results = [item["doc"] for item in reranked_docs[:k]]

    return final_results



In [ ]:
def query_rewrite(query):

  query_rewrite_instruction = f"""
  Nhiệm vụ của bạn là viết lại câu truy vấn (query) của người dùng để tối ưu hóa việc tìm kiếm trong Vector Database chứa nội dung bài giảng (bao gồm Slide và Lời giảng).

  ### Hướng dẫn thực hiện:
  1. **Phân tích ý định:** Xác định xem người dùng đang tìm kiếm thông tin trên Slide (dạng liệt kê, định nghĩa) hay trong Lời giảng (dạng giải thích, ví dụ).
  2. **Mở rộng thuật ngữ:** Thêm các từ đồng nghĩa hoặc thuật ngữ chuyên môn liên quan.
    - Ví dụ: "đồ án" -> "bài tập lớn", "project", "nhiệm vụ cuối kỳ".
    - Ví dụ: "giới thiệu môn học" -> "mục tiêu học tập", "đề cương bài giảng", "chuẩn đầu ra".
  3. **Cấu trúc lại:** Chuyển câu hỏi ngắn thành một đoạn mô tả súc tích chứa các từ khóa quan trọng mà cả Slide và Giảng viên có khả năng sẽ đề cập tới.
  4. **Ngôn ngữ:** Nếu query là tiếng Việt, hãy cung cấp phiên bản mở rộng bằng tiếng Việt (có thể kèm thuật ngữ tiếng Anh chuyên ngành nếu cần).

  ### Yêu cầu đầu ra:
  - Chỉ trả về nội dung câu query đã được viết lại.
  - Không kèm theo lời giải thích "Tôi đã viết lại là...".
  - Đảm bảo câu văn mang tính học thuật và sát với ngữ cảnh bài giảng.

  ### Đây là câu truy vấn ban đầu:
  {query}
"""

  refined_query = llm.invoke(query_rewrite_instruction).content
  return refined_query

refined_query = query_rewrite(query)
print(refined_query)

Tổng quan về môn học, mục tiêu học tập, đề cương chi tiết, chuẩn đầu ra, bài tập lớn, project, nhiệm vụ cuối kỳ, báo cáo đồ án.


In [ ]:
query = "giải thích về kiến trúc transformer"
refined_query = query_rewrite(query)
print(refined_query)

kiến trúc Transformer, mô hình Transformer, cơ chế Attention, Self-Attention, kiến trúc mạng neural, xử lý ngôn ngữ tự nhiên, giải thích chi tiết, nguyên lý hoạt động, ứng dụng của Transformer.


In [ ]:
results = hybrid_search(refined_query, vector_store, bm25_retriever, 10)

In [ ]:
context = ""
for i, res in enumerate(results, 1):
  context += f"##======Ket qua {i}======\n"
  context += f"###Video: {res.metadata.get("video_name")}\n"
  context += f"###Timestamp: {res.metadata.get("timestamp")} - Duration: {res.metadata.get('duration')}\n"
  context += f"{res.page_content}\n"
Markdown(context)

##======Ket qua 1======
###Video: [CS431 - Chương 10] Part 4_1_ Kiến trúc Transformer_ Bộ Encoder
###Timestamp: 180 - Duration: 40.0
### Nội dung Slide

*   **Transformer** là một loại mô hình máy học tiên tiến, được ứng dụng rộng rãi trong nhiều nhiệm vụ như xử lý ngôn ngữ tự nhiên, tự động hóa và phân loại dữ liệu.
*   Kiến trúc **Transformer** bao gồm hai thành phần chính:
    *   **Encoder**: Tiếp nhận dữ liệu đầu vào và chuyển đổi thành biểu diễn trung gian.
    *   **Decoder**: Tiếp nhận biểu diễn trung gian và tạo ra dữ liệu đầu ra.
*   Cấu trúc **Encoder** thường bao gồm nhiều lớp (ví dụ: 6 lớp), mỗi lớp lại chứa các **lõi** (lõi là đơn vị xử lý nhỏ nhất trong Transformer). (Lưu ý: Phần lặp lại "Encoder có 6 lớp, mỗi lớp có 6 lõi" trong slide gốc đã được tinh gọn).

### Lời giảng của Giảng viên

Khi chúng ta mới bắt đầu tìm hiểu, nhìn vào sơ đồ kiến trúc **Transformer** có thể gây rối vì có nhiều mô-đun và chúng ta chưa hiểu rõ mục đích của chúng.

Tại thời điểm này, chúng ta chỉ cần hình dung rằng **Transformer** bao gồm hai thành phần chính là **Encoder** và **Decoder**. Đây là **Encoder**, và đây là **Decoder**. Chúng ta sẽ đi sâu vào từng thành phần.

Đầu tiên, chúng ta sẽ tìm hiểu về **Encoder**. Mô-đun xử lý tính toán đầu tiên và quan trọng nhất trong **Encoder** chính là **Self-Attention**. **Self-Attention** là một **mô-đun cốt lõi** của **Transformer**. Dữ liệu đầu vào, dưới dạng từ hoặc token, sẽ được chuyển đổi thành vector biểu diễn thông qua **input embedding**. Vector biểu diễn này sau đó sẽ được đưa vào mô-đun **Self-Attention**.

**Self-Attention** hoạt động dựa trên cơ chế **Attention**. Cơ chế **Attention** có thể được hình dung tương tự như một phép **truy vấn** trong bảng dữ liệu, nơi chúng ta có một **query**.
##======Ket qua 2======
###Video: [CS431 - Chương 10] Part 2_ Động lực của kiến trúc Transformer
###Timestamp: 0 - Duration: 20.0
### Nội dung Slide

*   **Transformer** là một mô hình máy học được sử dụng rộng rãi trong các ứng dụng như:
    *   Tự động hóa ngôn ngữ
    *   Phân tích dữ liệu
    *   Xử lý ngôn ngữ tự nhiên (NLP)
*   **Transformer** được phát triển dựa trên các mô hình trước đó như BiLSTM, BiGRU, và BiLSTM-CRF.
*   Mô hình sử dụng các lớp **time-distributed layers** để xử lý các vấn đề về chuỗi thời gian dài.
*   Các lớp này giúp chuyển đổi các câu có độ dài lớn thành các chuỗi ngắn hơn, hiệu quả hơn.
*   Hình vẽ minh họa cho thấy cấu trúc **Transformer** với các lớp thời gian được áp dụng.

### Lời giảng của Giảng viên

Nội dung hôm nay chúng ta sẽ gồm ba phần. Đầu tiên, chúng ta sẽ cùng tìm hiểu về **kiến trúc Transformer**. Sau đó, chúng ta sẽ tìm hiểu về một số điểm yếu của **Transformer**. Và cuối cùng, là một số ứng dụng cũng như thành tựu.

Đầu tiên, về kiến trúc, chúng ta sẽ xét đến **động lực** – tại sao chúng ta cần phải có kiến trúc mạng **Transformer**? Động lực này xuất phát từ việc hai từ bất kỳ trong một đoạn văn bản đầu vào cần tương tác với nhau về mặt thông tin. Nếu chúng ta phải thực hiện nhiều thao tác để tính toán sự tương tác này, ví dụ giữa hai từ này, thì rõ ràng trong ngôn ngữ tự nhiên, các từ phải có sự liên hệ về mặt ý nghĩa với nhau. Chỉ khi đó, chúng ta mới có thể hiểu rõ nội dung của đầu vào và tính toán ra kết quả đầu ra phù hợp.
##======Ket qua 3======
###Video: [CS431 - Chương 10] Part 7_ Một số ứng dụng của kiến trúc mạng Transformer
###Timestamp: 20 - Duration: 40.0
### Nội dung Slide
*   **BERT**: Bidirectional Encoder Representation from Transformer
*   **GPT**: Generative Pre-trained Transformer

### Lời giảng của Giảng viên
Tiếp theo, chúng ta sẽ tìm hiểu về hướng ứng dụng của Transformer cũng như những thành tựu nổi bật, mà đầu tiên là hai mô hình nền tảng **BERT** và **GPT**. Đây là hai mô hình ngôn ngữ được huấn luyện trên tập dữ liệu lớn.

Trong kiến trúc Transformer, **BERT** tận dụng giai đoạn **Encoder**, còn **GPT** lại tập trung vào giai đoạn **Decoder**.

*   **BERT** là viết tắt của **Bidirectional Encoder Representation from Transformer**.
*   **GPT** là viết tắt của **Generative Pre-trained Transformer**. Từ "Generative" ở đây ám chỉ khả năng tạo sinh, tương tự như vai trò của **Decoder**.

Chúng ta thấy rằng cả hai mô hình này đều **dựa trên kiến trúc của Transformer**. Điểm chung quan trọng là cả hai đều thuộc nhóm **học tự giám sát (Self-supervised Learning)**, tức là chúng có khả năng học từ dữ liệu không cần gán nhãn.
##======Ket qua 4======
###Video: [CS431 - Chương 10] Part 4_1_ Kiến trúc Transformer_ Bộ Encoder
###Timestamp: 220 - Duration: 40.0
### Nội dung Slide

*   **Encoder: Self-Attention**
    *   **Self-Attention** là mô hình chính của Transformer.
*   Sơ đồ kiến trúc Transformer:
    *   **Encoder**:
        *   Input embedding
        *   Inputs
        *   **Self-Attention**
    *   **Decoder**:
        *   Output embedding
        *   Output (shifted right)

### Lời giảng của Giảng viên

Thì mô đun xử lý tính toán đầu tiên của **Encoder** chính là **Self-Attention**. **Self-Attention** là mô hình chính của Transformer.

Với dữ kiện đầu vào là các từ hoặc token, chúng ta sẽ chuyển chúng thành **vector biểu diễn** thông qua **Input Embedding**. Sau đó, các vector này sẽ đi đến mô đun **Self-Attention**.

**Self-Attention** dựa trên cơ chế **Attention**. Cơ chế **Attention** có thể hình dung tương đương với một phép truy vấn trong bảng dữ liệu. Trong đó, chúng ta có một **query** để tìm kiếm thông tin trong cơ sở dữ liệu, thông qua các **key** để trích xuất ra các **value** mong muốn.

Ví dụ thực tế cho cơ chế này là các hệ thống tìm kiếm:
*   **Query**: Từ khóa tìm kiếm (ví dụ: "transformer architecture").
*   **Key**: Tiêu đề của các video trên YouTube.
*   **Value**: Nội dung, mô tả của video.

Tuy nhiên, **Attention** trong Transformer khác với truy vấn bảng dữ liệu thông thường ở chỗ: mỗi **query** sẽ khớp với **tất cả các key** và trả về tổng hợp của **tất cả các value** tương ứng, thay vì chỉ trích xuất một giá trị duy nhất.
##======Ket qua 5======
###Video: [CS431 - Chương 10] Part 2_ Động lực của kiến trúc Transformer
###Timestamp: 20 - Duration: 40.0
### Nội dung Slide

*   **Transformers** là một loại mô hình máy học được sử dụng rộng rãi trong các ứng dụng như **tự động hóa ngôn ngữ**, phân tích dữ liệu, và **xử lý ngôn ngữ tự nhiên (NLP)**.
*   **Động lực** ra đời của Transformer xuất phát từ việc cải thiện khả năng xử lý mối quan hệ giữa các từ trong một chuỗi dữ liệu dài.
*   Các mô hình trước đó như BiLSTM, BiGRU gặp hạn chế trong việc nắm bắt mối quan hệ xa.
*   Transformer sử dụng cơ chế **Attention** để cho phép các từ tương tác trực tiếp với nhau, bất kể khoảng cách.
*   Điều này giúp **tối thiểu hóa độ dài của sự tương tác** giữa các từ và **tối đa hóa thao tác song song**, khai thác hiệu quả sức mạnh của phần cứng tính toán.

### Lời giảng của Giảng viên

Đầu tiên, chúng ta sẽ cùng tìm hiểu về **kiến trúc Transformer**. Chúng ta sẽ xét đến **động lực** tại sao chúng ta cần có kiến trúc mạng **Transformer**.

Động lực này xuất phát từ việc, khi xét hai từ bất kỳ trong một đoạn input, nếu chúng ta muốn chúng tương tác với nhau về mặt thông tin, chúng ta sẽ tốn rất nhiều thao tác. Trong ngôn ngữ tự nhiên, các từ luôn có sự liên hệ về mặt ý nghĩa với nhau để chúng ta hiểu rõ nội dung.

Nếu không có **module attention** này, sự tương tác giữa hai từ bất kỳ trong một câu sẽ phải đi qua tất cả các từ khác trong câu. Tức là độ dài tương tác sẽ phụ thuộc vào độ dài của câu (SQLEN). Chúng ta sẽ nói kỹ hơn về điều này sau.

Do đó, động lực của Transformer là làm sao để **tối thiểu hóa độ dài của sự tương tác** giữa hai từ bất kỳ trong câu. Đồng thời, chúng ta cần **tối đa hóa thao tác song song**, vì **Deep Learning** muốn hiệu quả thì phải khai thác được sức mạnh của các thiết bị tính toán song song.
##======Ket qua 6======
###Video: [CS431 - Chương 10] Part 4_1_ Kiến trúc Transformer_ Bộ Encoder
###Timestamp: 0 - Duration: 20.0
### Nội dung Slide

*   **Transformer**
*   **Ứng dụng của Transformer:**
    *   Ngôn ngữ song song không phụ thuộc chi tiết đại chi tiết.
    *   Môi trường tương tác với các từ còn lại với **O(1)** bước.

### Lời giảng của Giảng viên

Trong các kiến trúc cũ như **Recurrent Neural Network (RNN)**, quá trình tính toán thông tin diễn ra một cách **tuần tự**. Nghĩa là, để xử lý đến từ cuối cùng trong giai đoạn **encode**, chúng ta phải lan truyền thông tin tuần tự qua từng bước. Điều này tốn rất nhiều bước xử lý tuần tự.

Hậu quả là, thông tin của từ đầu tiên khi đến được từ cuối cùng có thể đã bị **mất mát đáng kể**. Đây chính là điểm yếu của RNN. Hơn nữa, việc xử lý tuần tự này cũng **không tận dụng được sức mạnh xử lý song song của GPU**.

Kiến trúc **Transformer**, nhờ cơ chế **Self-Attention**, cho phép xử lý song song. Khi tính toán tại một vị trí, chúng ta không cần phụ thuộc vào kết quả tính toán tại các vị trí khác trong cùng một layer. Các node trên cùng một layer được tính toán **độc lập với nhau**.

Ngược lại, với RNN, để tính toán tại một vị trí, chúng ta phải hoàn thành tính toán ở các vị trí trước đó. Việc các node tính toán độc lập cho phép chúng ta **sử dụng hiệu quả GPU**. Do đó, các phép tính song song của Transformer **không phụ thuộc vào chiều dài của chuỗi**, ngay cả khi chuỗi rất dài, chúng vẫn có thể thực hiện song song.
##======Ket qua 7======
###Video: [CS431 - Chương 10] Part 4_1_ Kiến trúc Transformer_ Bộ Encoder
###Timestamp: 20 - Duration: 80.0
### Nội dung Slide
*   **Transformer:** Ứng dụng của **Transformer**
    *   Sơ đồ tổng quát về **xử lý ngôn ngữ song song**
    *   Môi trường tương tác với các từ còn lại trong **O(1) bước**

### Lời giảng của Giảng viên
Trước đây, với các kiến trúc cũ như **Recurrent Neural Network (RNN)**, quá trình tính toán phải thực hiện một cách tuần tự. Điều này có nghĩa là, để xử lý đến từ cuối cùng trong giai đoạn **encode**, thông tin phải lan truyền tuần tự qua từng bước. Cách tiếp cận này tốn rất nhiều bước xử lý tuần tự, và thông tin từ các từ đầu tiên có thể bị suy giảm đáng kể khi đến được các từ cuối. Đây chính là điểm yếu của **RNN**. Hơn nữa, việc xử lý tuần tự này không tận dụng được sức mạnh xử lý song song của **GPU**.

Kiến trúc **Transformer**, nhờ cơ chế **Self-Attention**, cho phép xử lý song song. Khi tính toán tại một vị trí, ta không cần phụ thuộc vào các giá trị đã được tính toán trước đó. Các **node** trên cùng một **layer** có thể được thực hiện một cách độc lập với nhau. Ngược lại, với **RNN**, để tính toán tại một **hidden state**, ta phải tính toán trước đó rồi mới đến được vị trí hiện tại. Các **node** độc lập có thể tận dụng **GPU**, do đó, số phép tính song song không phụ thuộc vào chiều dài của chuỗi. Ngay cả khi chuỗi rất dài, **Transformer** vẫn có thể thực hiện song song.

Các kết nối dày đặc trong **Transformer** cho phép tương tác giữa các từ. Nếu ở **RNN**, để tương tác giữa từ đầu tiên và từ cuối cùng cần nhiều bước tuần tự, thì trong **Transformer**, từ đầu tiên và từ cuối cùng của lớp trước đó có thể tương tác trực tiếp thông qua các lớp tiếp theo, mà không cần lan truyền tuần tự. Ví dụ, tại **layer 2**, thông tin từ **layer 1** đã có thể truy xuất trực tiếp thông tin của từ đầu tiên và từ cuối cùng. Đây chính là những ưu điểm vượt trội của **Transformer**. Hình vẽ trên là sơ đồ kiến trúc tổng quát của **Transformer**, ban đầu có thể gây rối với nhiều mô-đun, nhưng chúng ta sẽ tìm hiểu lý do tại sao lại có những mô-đun này.
##======Ket qua 8======
###Video: [CS431 - Chương 10] Part 1_ Ôn tập kiến trúc mạng RNN và cơ chế Attention
###Timestamp: 20 - Duration: 20.0
### Nội dung Slide

*   **CS431 - CÁC KỸ THUẬT HỌC SÂU VÀ ỨNG DỤNG**
*   **BÀI 10**
*   **TRANSFORMER VÀ ỨNG DỤNG**

### Lời giảng của Giảng viên

Chào các bạn, hôm nay chúng ta sẽ cùng đến với bài **Transformer** và một số ứng dụng của Transformer trong xử lý ngôn ngữ tự nhiên. Đây có thể nói là một trong những bài rất quan trọng, nó sẽ là nền tảng để chúng ta học tiếp những thành tựu của Transformer trong các lĩnh vực khác, không chỉ trong xử lý ngôn ngữ tự nhiên mà còn có thể dùng trong lĩnh vực về hình ảnh, xử lý âm thanh.

Nội dung bài hôm nay chúng ta sẽ cùng:

1.  Ôn tập lại một số khái niệm, ví dụ như:
    *   **Attention** là gì?
    *   Cách tính **Attention Score**, **Attention Distribution** và **Attention Output**.
2.  Trong hình ở đây, chúng ta thấy đó là...
##======Ket qua 9======
###Video: [CS431 - Chương 10] Part 7_ Một số ứng dụng của kiến trúc mạng Transformer
###Timestamp: 240 - Duration: 60.0
### Nội dung Slide

*   **Các mô hình ngôn ngữ lớn: BERT và GPT**
*   **Điểm khác biệt chính:**
    *   **BERT:**
        *   **Mô hình:** Mô hình ngôn ngữ **ngụy trang (Masked)**.
        *   **Cấu tạo:** Bao gồm các khối **Encoder** trong kiến trúc **Transformer**.
        *   **Nhiệm vụ:** Dự đoán **từ bị che (masked word)**.
        *   **Các tác vụ hạ nguồn (Downstream tasks) phù hợp:** **Phân loại văn bản**, **trả lời câu hỏi (Question Answering)**, **tóm tắt bài viết (Summarization)**, **nhận diện thực thể có tên (Named Entity Recognition)**.
    *   **GPT:**
        *   **Mô hình:** Mô hình ngôn ngữ **tự hồi quy (Autoregressive)**.
        *   **Cấu tạo:** Bao gồm các khối **Decoder** trong kiến trúc **Transformer**.
        *   **Nhiệm vụ:** Dự đoán **từ tiếp theo (next word)**.
        *   **Các tác vụ hạ nguồn (Downstream tasks) phù hợp:** **Dịch máy (Machine Translation)**, **tạo sinh nội dung tự động (Content Generation)**.

### Lời giảng của Giảng viên

Chúng ta sẽ dự đoán từ tiếp theo. Còn mô hình ngôn ngữ ẩn, tức là chúng ta sẽ che đi một từ ở giữa, một từ bất kỳ, một từ ngẫu nhiên, và chúng ta sẽ phải đoán xem từ bị che đó là gì. Đó là hai cái điểm khác biệt về cách huấn luyện.

Về cấu tạo thì BERT bao gồm các khối **Encoder** trong Transformer, còn GPT thì được cấu tạo bởi các khối **Decoder** trong Transformer. Nhiệm vụ của BERT là **che từ**, tức là đoán từ bị che (**masked word**), còn GPT sẽ là đoán từ tiếp theo (**next word**).

Đối với các **downstream tasks** mà mô hình BERT phù hợp có thể kể đến như **phân loại văn bản**, **trả lời câu hỏi**, **tóm tắt văn bản**, hoặc **nhận diện thực thể có tên (Named Entity Recognition)**. Còn các **downstream tasks** cho GPT mà nó khá phù hợp đó chính là **dịch máy** và **tạo sinh nội dung tự động**.

Và để sử dụng hai cái mô hình nền tảng này, chúng ta sẽ tìm hiểu một số cách. Cách đầu tiên đó là **fine-tuning**. **Fine-tuning** hiểu một cách nôm na đó chính là chúng ta sẽ huấn luyện lại hoặc là chúng ta sẽ thay đổi các tham số của mô hình.
##======Ket qua 10======
###Video: [CS431 - Chương 10] Part 4_1_ Kiến trúc Transformer_ Bộ Encoder
###Timestamp: 100 - Duration: 80.0
### Nội dung Slide

*   **Transformer**: Ứng dụng của Transformer
    *   Sơ đồ tổng quát của **Transformer**.
    *   Môi trường tương tác với các từ con liên tục với **O(1) bước**.

### Lời giảng của Giảng viên

Do đó, mỗi phép tính song song của chúng ta sẽ không phụ thuộc vào chiều dài của chuỗi. Nghĩa là, khi chuỗi rất dài, chúng ta vẫn có thể thực hiện song song.

Đồng thời, các kết nối dày đặc này cho phép chúng ta tương tác giữa các từ. Nếu ở các mô hình trước, để tương tác giữa từ đầu tiên và từ cuối cùng cần rất nhiều bước tuần tự, thì trong **Transformer**, từ đầu tiên và từ cuối cùng đã có thể tương tác trực tiếp thông qua các lớp trước đó. Cụ thể, ở lớp số 2, chúng ta có thể truy xuất thông tin từ lớp số 1, bao gồm cả thông tin của từ đầu tiên và từ cuối cùng một cách trực tiếp mà không cần thực hiện tuần tự.

Đây chính là những ưu điểm của **Transformer**. Hình vẽ trên là sơ đồ kiến trúc tổng quát của **Transformer**. Khi mới nhìn vào, chúng ta có thể cảm thấy rối vì có nhiều mô-đun và chưa hiểu rõ mục đích của chúng.

Tuy nhiên, ngay lúc này, chúng ta chỉ cần hình dung **Transformer** bao gồm hai thành phần chính: **Encoder** và **Decoder**. Đây là **Encoder** và đây là **Decoder**. Chúng ta sẽ cùng đi vào từng thành phần. Đầu tiên là **Encoder**. Mô-đun xử lý tính toán đầu tiên của **Encoder** chính là **Self-Attention**. **Self-Attention** là một mô-đun cốt lõi của **Transformer**.


## Retrieval Strategy (Workflow)

### Build bang LangGraph (Dang lam)

In [ ]:
from langgraph.graph import StateGraph, END